# [African Bird Club](https://www.africanbirdclub.org/) Data

The first source of data for this project was from the African Bird Club website. The website contains 26,057 images of 2369 different bird species as of 5th February 2025, from different countries of Africa.

## Introduction to the African Bird Club (ABC)
The African Bird Club (ABC) is a non-governmental organization dedicated to the conservation and study of birds across Africa. Founded in 1986, ABC works to promote the appreciation, protection, and study of birdlife on the African continent. They bring together birdwatchers, conservationists, and researchers who share a passion for preserving Africa's rich and diverse bird species.

### Mission and Vision
ABC’s mission is to support and advance the conservation of African bird species and their habitats. Through a variety of programs, they engage with local communities, birdwatching groups, conservation organizations, and researchers to create a deep understanding of Africa’s avian biodiversity. The club’s vision extends to making Africa a place where birds, their habitats, and the people who depend on them are protected and celebrated.

### Key Areas of Focus
- **Bird Conservation:** ABC works to safeguard habitats and ensure that bird populations across Africa are protected. They provide valuable data for bird conservation programs and work closely with local communities to implement conservation actions.

- **Research and Education:** The club encourages scientific research and educational initiatives that increase the understanding of African birds. Their members contribute to research efforts on migration, breeding patterns, and conservation needs.

- **Birdwatching Community:** ABC promotes birdwatching as a tool for both enjoyment and education. They provide resources and opportunities for birdwatchers and researchers to share information and observations about Africa's bird species.

- **Publications:** One of ABC's key contributions is their publication of the African Bird Club Bulletin, a quarterly magazine that shares research, birding reports, and conservation updates.

## How ABC Will Help My Project
As part of my bird pest control project, the African Bird Club (ABC) plays an important role in providing a comprehensive database of bird species found across Africa. ABC’s resources, including species lists, detailed bird profiles, and high-quality bird images, and their locations, will be utilized in training my computer vision model for detecting bird pests in agricultural settings.

### Using ABC’s Resources for Web Scraping and Image Collection
By utilizing the images and species data provided by ABC, I can focus on specific bird species that are prevalent in agricultural environments, allowing me to tailor my bird pest control model to accurately identify these species in real-world agricultural scenarios. ABC’s online resources and database will help me collect and organize bird images necessary for building the dataset needed for training the model.

In this notebook, I will employ web scraping techniques to gather bird images from the ABC website, relevant to my project’s goal of detecting and managing bird pest impacts on crops. The images collected will be processed and categorized for model training, contributing to the overall success of the Bird Detection and Pest Control project.

## Importing Libraries

Below are all the libraries that shall be used for collecting data from the African Bird Club Website.

In [1]:
from bs4 import BeautifulSoup
import requests
import time

import pandas as pd

import shutil
import os

from tqdm import tqdm

## Web Scraping
The first thing I did was set the links to the [website](https://www.africanbirdclub.org/) that we would be accessing information from. I separated the two links to keep the code clean and flexible since I shall be collecting information from different pages. 
- The `base_url` holds the main address of the site and serves as the foundation for any additional links on the site
- `species_info_url` contains the path to the page with the kkenyan bird species information. 

This approach makes it easier to update or reuse parts of the URL later on if needed.

In [2]:
# Set the links to the pages to retrieve information from
abc_base_url = 'https://www.africanbirdclub.org'
abc_species_url = f'{abc_base_url}/afbid/search/category/-/-/-'

When scraping data from websites, it’s important to identify your request as coming from a legitimate source, such as a web browser rather than a bot or scraper, so as to allow one to retrieve the necessary data smoothly. Websites can block requests if they think the traffic isn’t from a real user. To avoid this, we need to set headers that make the requests look like they're coming from a browser. 

I set the headers to include a `User-Agent` (more information about `User-Agent` in [User-Agent Documentation](https://pypi.org/project/user-agents/)), which is a string used by web browsers to identify themselves when making requests to websites. By using this, we ensure that the website allows our code to access the information without blocking it. In this case, I used a User-Agent that mimics a standard Chrome browser on Windows.

To understand more about why headers are important you can read [this article.](https://www.zenrows.com/blog/python-requests-user-agent#what-is)

In [3]:
# Set headers with a User-Agent to allow access
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36'
}

Before scraping, always review the `robots.txt` file to ensure that you are scraping content that is allowed and ethical to collect, as the file provides specific instructions on which parts of the website can be crawled and indexed and which should be avoided. 

I sent a request using the `requests.get` method, which sends a GET request to the `robots_url`, containing the specifications for web crawlers. I also include the headers  defined earlier to ensure the request is accepted. You can find more details about the requests.get method in the [Requests Documentation](https://pypi.org/project/requests/).

In [4]:
# Set the URL for the robots.txt file
robots_url = 'https://www.africanbirdclub.org/robots.txt'

# Fetch the robots.txt file
response = requests.get(robots_url, headers=headers)

print(response.text)

User-agent: *
Disallow: /wp-content/uploads/wc-logs/
Disallow: /wp-content/uploads/woocommerce_transient_files/
Disallow: /wp-content/uploads/woocommerce_uploads/
Disallow: /wp-admin/
Allow: /wp-admin/admin-ajax.php

# START YOAST BLOCK
# ---------------------------
User-agent: *
Disallow:

Sitemap: https://www.africanbirdclub.org/sitemap_index.xml
# ---------------------------
# END YOAST BLOCK


The site doesn't explicitly restrict any particular content from being scraped, except for certain admin areas and temporary or log files. This means we are free to scrape content that is publicly accessible on the website (like bird species data, provided it's not behind authentication or a paywall).

After checking that we are within the ethical and legal considerations of the website, I sent a request using the `requests.get` method, which sends a GET request to the `species_info_url`, containing the specific endpoint for the bird species information. 

In [5]:
# Send a request to the species list page
response_info = requests.get(abc_species_url, headers=headers)
if response_info.status_code != 200:
    print(f"Failed to retrieve species list page, status code: {response_info.status_code}")
else:
    soup_info = BeautifulSoup(response_info.content, 'html.parser')

In the next step, the HTML structure of the page is assessed. It can be noted that the structure is such that the bird species are arranged in a list contained in the `<div>` class `panel-inner`. 

![Species Page HTML structure](images_used/species-list.png)

To know the number of species on the web page, whose images we will be using:

In [6]:
# Find all the species elements
species_elements = soup_info.find_all("li")

# Count the number of species
species_count = len(species_elements)

print(f"Number of species: {species_count}")

Number of species: 2393


Next, we target a specific section of the webpage where the species information is contained. The list of species is located within a `<div>` with a class of "panel-inner", and inside it, there's an unordered list `(<ul>)` with the class "type" that holds the individual species.

In [7]:
# Find the div and ul containing the species list
species_div = soup_info.find('div', class_='panel-inner')  # Find the div with class "panel-inner"
species_list = species_div.find('ul', class_='type')  # Find the ul with class "type"

After setting up the location where the list of the different species is stored, we loop through each species item to extract detailed information about the species. For each species, we gather the scientific name, common name, image count, and image link information on each species.

The extracted information is then stored in a list of tuples called `abc_birds`. Each tuple contains the data for one species.

In [8]:
# Find all species items (li elements)
species_items = species_list.find_all('li')

# Extract information (scientific_name, common_name, image_count, image_link) on the bird species
abc_birds = []  # List to store information in tuples

for species_item in species_items:
    # Get the species info
    scientific_name = species_item.find('h5').text.strip().split(' (')[0]  # Get the scientific name
    common_name = species_item.find('span').text.strip()  # Get the common name
    image_count = int(species_item.find("h5").text.split('(')[-1].split(')')[0]) # Get the image count
    image_link = species_item.find("a")['href'] # Get the image link
    
    # Store the species and common names in the list
    abc_birds.append((scientific_name, common_name, image_count, image_link))

Once we have successfully extracted the species information, we can output the total number of species we’ve collected and display a sample of the data.

This provides a quick check of the data extracted to ensure it looks correct and matches expectations.

In [9]:
# Print the total number of species extracted
print(f"Total number of species extracted: {len(abc_birds)}")

# Print the first 5 species as a sample
print("Sample of extracted species:")
for bird in abc_birds[:5]:  # Limit to the first 5 species
    print(bird)

Total number of species extracted: 2369
Sample of extracted species:
('Accipiter henstii', 'Henst’s Goshawk', 4, '/afbid/search/browse/species/290')
('Accipiter madagascariensis', 'Madagascar Sparrowhawk', 5, '/afbid/search/browse/species/285')
('Accipiter nisus', 'Eurasian Sparrowhawk', 6, '/afbid/search/browse/species/286')
('Accipiter ovampensis', 'Ovambo Sparrowhawk', 8, '/afbid/search/browse/species/284')
('Accipiter rufiventris', 'Rufous-breasted Sparrowhawk', 4, '/afbid/search/browse/species/287')


In [10]:
abc_birds_df = pd.DataFrame(abc_birds, columns=['Scientific_Name', 'Common_Name', 'Image_Count', 'Image_Link' ])
abc_birds_df.head()

,Scientific_Name,Common_Name,Image_Count,Image_Link
0,Accipiter henstii,Henst’s Goshawk,4,/afbid/search/browse/species/290
1,Accipiter madagascariensis,Madagascar Sparrowhawk,5,/afbid/search/browse/species/285
2,Accipiter nisus,Eurasian Sparrowhawk,6,/afbid/search/browse/species/286
3,Accipiter ovampensis,Ovambo Sparrowhawk,8,/afbid/search/browse/species/284
4,Accipiter rufiventris,Rufous-breasted Sparrowhawk,4,/afbid/search/browse/species/287


In [11]:
abc_birds_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2369 entries, 0 to 2368
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Scientific_Name  2369 non-null   object
 1   Common_Name      2369 non-null   object
 2   Image_Count      2369 non-null   int64 
 3   Image_Link       2369 non-null   object
dtypes: int64(1), object(3)
memory usage: 74.2+ KB


In [12]:
# Function to download images in batches
def download_abc_images(df, start_idx, end_idx):

    # Loop over the specified range of species
    for i in range(start_idx, end_idx):
        abc_birds_info = df.iloc[i]
        images_page_url = f"{abc_base_url}{abc_birds_info['Image_Link']}"
        common_name = abc_birds_info['Common_Name']

        # Request species page
        response_species = requests.get(images_page_url, headers=headers)
        if response_species.status_code != 200:
            print(f"Failed to retrieve species page for {common_name}, status code: {response_species.status_code}")
            continue
        
        soup_species = BeautifulSoup(response_species.content, 'html.parser')
        # Find and download images
        image_list = soup_species.find('ul', class_='row image-list')
        if image_list:
            images = image_list.findAll('img')
            for idx, img in enumerate(images):
                img_src = img.attrs['src']
                img_url = f"{abc_base_url}{img_src}"
                
                # Create folder for the species
                images_folder = f"ABC_BirdImages/{common_name}"
                if not os.path.exists(images_folder):
                    os.makedirs(images_folder)

                # Download and save image
                img_response = requests.get(img_url, headers=headers, stream=True, timeout=10)
                if img_response.headers['Content-Type'].startswith('image/'):
                    img_filename = f"{images_folder}/{common_name}_img{idx+1}.jpg"
                    with open(img_filename, 'wb') as f:
                        shutil.copyfileobj(img_response.raw, f)
                    print(f"Downloaded: {img_filename}")
                else:
                    print(f"Unexpected content type for {common_name}: {img_response.headers['Content-Type']}")

        # Sleep to avoid overwhelming the server
        time.sleep(5)


In [13]:
df = abc_birds_df
download_abc_images(df, 0, 5)

Downloaded: ABC_BirdImages/Henst’s Goshawk/Henst’s Goshawk_img1.jpg
Downloaded: ABC_BirdImages/Henst’s Goshawk/Henst’s Goshawk_img2.jpg
Downloaded: ABC_BirdImages/Henst’s Goshawk/Henst’s Goshawk_img3.jpg
Downloaded: ABC_BirdImages/Henst’s Goshawk/Henst’s Goshawk_img4.jpg


KeyboardInterrupt: 

In [14]:
# Define batch size
batch_size = 5

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-100, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches of 10, starting from the 101th species (index 101)
    for start_idx in range(100, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking
        
        download_abc_images(abc_birds_df, start_idx, end_idx)  # Download batch
        
        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar

tqdm.write("✅ All images downloaded successfully!")


📥 Downloading images for species 101 to 105...
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img1.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img2.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img3.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img4.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img5.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img6.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img7.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img8.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img9.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img10.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img11.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Duck_img12.jpg
Downloaded: ABC_BirdImages/African Black Duck/African Black Du

📥 Downloading images for species 106 to 110...
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img1.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img2.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img3.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img4.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img5.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img6.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img7.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img8.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img9.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img10.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img11.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img12.jpg
Downloaded: ABC_BirdImages/Cuckoo-finch/Cuckoo-finch_img13.jpg
Downloaded: ABC_BirdImages/Black Noddy/Black Noddy_img1.jpg
Downloaded: ABC_BirdImages/Black Noddy/Black Noddy_img2.jpg
Downloaded: ABC_BirdIma

📥 Downloading images for species 111 to 115...
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img1.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img2.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img3.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img4.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img5.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img6.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img7.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img8.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img9.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img10.jpg
Downloaded: ABC_BirdImages/Orange-breasted Sunbird/Orange-breasted Sunbird_img11.jpg
Downloaded: ABC_BirdImages/

📥 Downloading images for species 116 to 120...
Downloaded: ABC_BirdImages/Yellow Penduline Tit/Yellow Penduline Tit_img1.jpg
Downloaded: ABC_BirdImages/Yellow Penduline Tit/Yellow Penduline Tit_img2.jpg
Downloaded: ABC_BirdImages/Yellow Penduline Tit/Yellow Penduline Tit_img3.jpg
Downloaded: ABC_BirdImages/Yellow Penduline Tit/Yellow Penduline Tit_img4.jpg
Downloaded: ABC_BirdImages/Yellow Penduline Tit/Yellow Penduline Tit_img5.jpg
Downloaded: ABC_BirdImages/Sennar Penduline Tit/Sennar Penduline Tit_img1.jpg
Downloaded: ABC_BirdImages/Sennar Penduline Tit/Sennar Penduline Tit_img2.jpg
Downloaded: ABC_BirdImages/Sennar Penduline Tit/Sennar Penduline Tit_img3.jpg
Downloaded: ABC_BirdImages/Sennar Penduline Tit/Sennar Penduline Tit_img4.jpg
Downloaded: ABC_BirdImages/Sennar Penduline Tit/Sennar Penduline Tit_img5.jpg
Downloaded: ABC_BirdImages/Sennar Penduline Tit/Sennar Penduline Tit_img6.jpg
Downloaded: ABC_BirdImages/Anchieta’s Sunbird/Anchieta’s Sunbird_img1.jpg
Downloaded: ABC_BirdI

📥 Downloading images for species 121 to 125...
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img1.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img2.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img3.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img4.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img5.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img6.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img7.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img8.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_img9.jpg
Downloaded: ABC_BirdImages/Western Violet-backed Sunbird/Western Violet-backed Sunbird_im

...

📥 Downloading images for species 416 to 420...
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img5.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img6.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img7.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img8.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img9.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img10.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img11.jpg
Downl

📥 Downloading images for species 421 to 425...
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img5.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img6.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img7.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img8.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img9.jpg
Downloaded: ABC_BirdImages/Montane Nightjar/Montane Nightjar_img10.jpg
Unexpected content type for Red-necked Nightjar: text/html; charset=UTF-8
Downloaded: ABC_BirdImages/Red-necked Nightjar/Red-necked Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Red-necked Nightjar/Red-necked Nightjar_img3.jpg
Downloaded: ABC_BirdImages

📥 Downloading images for species 426 to 430...
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img5.jpg
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img6.jpg
Downloaded: ABC_BirdImages/Freckled Nightjar/Freckled Nightjar_img7.jpg
Downloaded: ABC_BirdImages/Pennant-winged Nightjar/Pennant-winged Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Pennant-winged Nightjar/Pennant-winged Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Pennant-winged Nightjar/Pennant-winged Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Pennant-winged Nightjar/Pennant-winged Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Pennant-winged Nightjar/Pennant-winged Nightjar_img5.jpg
Downloaded: ABC_BirdImages/Pe

📥 Downloading images for species 431 to 435...
Downloaded: ABC_BirdImages/Madagascar Cuckooshrike/Madagascar Cuckooshrike_img1.jpg
Downloaded: ABC_BirdImages/Madagascar Cuckooshrike/Madagascar Cuckooshrike_img2.jpg
Downloaded: ABC_BirdImages/Madagascar Cuckooshrike/Madagascar Cuckooshrike_img3.jpg
Downloaded: ABC_BirdImages/Madagascar Cuckooshrike/Madagascar Cuckooshrike_img4.jpg
Downloaded: ABC_BirdImages/Madagascar Cuckooshrike/Madagascar Cuckooshrike_img5.jpg
Downloaded: ABC_BirdImages/Madagascar Cuckooshrike/Madagascar Cuckooshrike_img6.jpg


ConnectionError: ('Connection aborted.', TimeoutError(10060, 'A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond', None, 10060, None))

In [ ]:
# Define batch size
batch_size = 5
timeout_duration = 5 * 60  # 5 minutes in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-400, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 401st species (index 400)
    for start_idx in range(400, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking
        
        download_abc_images(abc_birds_df, start_idx, end_idx)  # Download batch
        
        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 5-minute break after every two batches (10 species)
        if (start_idx - 400 + batch_size) % 10 == 0:  
            tqdm.write("⏳ Taking a 5-minute break...")
            time.sleep(timeout_duration)
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")


📥 Downloading images for species 401 to 405...
Downloaded: ABC_BirdImages/Tullberg's Woodpecker/Tullberg's Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Tullberg's Woodpecker/Tullberg's Woodpecker_img2.jpg
Downloaded: ABC_BirdImages/Tullberg's Woodpecker/Tullberg's Woodpecker_img3.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img1.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img2.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img3.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img4.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img5.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img6.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img7.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img8.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/Buff-streaked Chat_img9.jpg
Downloaded: ABC_BirdImages/Buff-streaked Chat/B

📥 Downloading images for species 406 to 410...
Downloaded: ABC_BirdImages/Grey-throated Rail/Grey-throated Rail_img1.jpg
Downloaded: ABC_BirdImages/Egyptian Nightjar/Egyptian Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Egyptian Nightjar/Egyptian Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Egyptian Nightjar/Egyptian Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img5.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img6.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_img7.jpg
Downloaded: ABC_BirdImages/Slender-tailed Nightjar/Slender-tailed Nightjar_

⏳ Taking a 5-minute break...


✅ Resuming downloads...
📥 Downloading images for species 411 to 415...
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img1.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img2.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img3.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img4.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img5.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img6.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img7.jpg
Downloaded: ABC_BirdImages/European Nightjar/European Nightjar_img8.jpg
Downloaded: ABC_BirdImages/Golden Nightjar/Golden Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Golden Nightjar/Golden Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Golden Nightjar/Golden Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Golden Nightjar/Golden Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Golden Nightjar/Golden Nightjar_img5.jpg
Downloaded: A

📥 Downloading images for species 416 to 420...
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img1.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img2.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img3.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img4.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img5.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img6.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img7.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img8.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img9.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img10.jpg
Downloaded: ABC_BirdImages/Standard-winged Nightjar/Standard-winged Nightjar_img11.jpg
Downl

...

✅ Resuming downloads...
📥 Downloading images for species 511 to 515...
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img2.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img3.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img4.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img5.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img6.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img7.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img8.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img9.jpg
Downloaded: ABC_BirdImages/Yellow-crested Woodpecker/Yellow-crested Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Yellow-crested Woodpecker/Yellow-crested Woodpecker_img2.j

📥 Downloading images for species 516 to 520...
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img1.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img2.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img3.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img4.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img5.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img6.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img7.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img8.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img9.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img1.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img2.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img3.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img4.jpg
Downloaded: ABC_BirdImages/Dieder

⏳ Taking a 5-minute break...


✅ Resuming downloads...
📥 Downloading images for species 521 to 525...
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img1.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img2.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img3.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img4.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img5.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img6.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img7.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img8.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img9.jpg
Downloaded: ABC_BirdImages/Collared Palm Thrush/Collared Palm Thrush_img10.jpg
Downloaded: ABC_BirdImages/Spotted Palm Thrush/Spotted Palm Thrush_img1.jpg
Downloaded: ABC_BirdImages/Spotted Palm Thrush/Spotted Palm Thrush_img2.

📥 Downloading images for species 526 to 530...
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img1.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img2.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img3.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img4.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img5.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img6.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img7.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img8.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img9.jpg
Downloaded: ABC_BirdImages/African Woolly-necked Stork/African Woolly-necked Stork_img10.jpg
Downloaded: ABC_BirdImages/Afric

In [ ]:
import time
import requests
from tqdm import tqdm

# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-500, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 501st species (index 500)
    for start_idx in range(500, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking

        try:
            download_abc_images(abc_birds_df, start_idx, end_idx)  # No timeout here!
        except Exception as e:
            tqdm.write(f"❌ Error: {e} - Skipping species {start_idx+1} to {end_idx}")

        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 500 + batch_size) % 10 == 0:
            tqdm.write("⏳ Taking a 1-minute break...")
            
            # Sleep in small chunks so it's interruptible
            for _ in range(timeout_duration):  
                time.sleep(1)  # Allows interruptions every second
            
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")


📥 Downloading images for species 501 to 505...
Downloaded: ABC_BirdImages/Falkenstein's Greenbul/Falkenstein's Greenbul_img1.jpg
Downloaded: ABC_BirdImages/Falkenstein's Greenbul/Falkenstein's Greenbul_img2.jpg
Downloaded: ABC_BirdImages/Falkenstein's Greenbul/Falkenstein's Greenbul_img3.jpg
Downloaded: ABC_BirdImages/Falkenstein's Greenbul/Falkenstein's Greenbul_img4.jpg
Downloaded: ABC_BirdImages/Falkenstein's Greenbul/Falkenstein's Greenbul_img5.jpg
Downloaded: ABC_BirdImages/Yellow-bellied Greenbul/Yellow-bellied Greenbul_img1.jpg
Downloaded: ABC_BirdImages/Yellow-bellied Greenbul/Yellow-bellied Greenbul_img2.jpg
Downloaded: ABC_BirdImages/Yellow-bellied Greenbul/Yellow-bellied Greenbul_img3.jpg
Downloaded: ABC_BirdImages/Yellow-bellied Greenbul/Yellow-bellied Greenbul_img4.jpg
Downloaded: ABC_BirdImages/Yellow-bellied Greenbul/Yellow-bellied Greenbul_img5.jpg
Downloaded: ABC_BirdImages/Yellow-bellied Greenbul/Yellow-bellied Greenbul_img6.jpg
Downloaded: ABC_BirdImages/Yellow-belli

📥 Downloading images for species 506 to 510...
Downloaded: ABC_BirdImages/Mount Kupe Bushshrike/Mount Kupe Bushshrike_img1.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img1.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img2.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img3.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img4.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img5.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img6.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img7.jpg
Downloaded: ABC_BirdImages/Black-fronted Bushshrike/Black-fronted Bushshrike_img8.jpg
Downloaded: ABC_BirdImages/Olive Bushshrike/Olive Bushshrike_img1.jpg
Downloaded: ABC_BirdImages/Olive Bushshrike/Olive Bushshrike_img2.jpg
Downloaded: ABC_BirdImages/Olive Bushshrike/O

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 511 to 515...
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img2.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img3.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img4.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img5.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img6.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img7.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img8.jpg
Downloaded: ABC_BirdImages/Fire-bellied Woodpecker/Fire-bellied Woodpecker_img9.jpg
Downloaded: ABC_BirdImages/Yellow-crested Woodpecker/Yellow-crested Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Yellow-crested Woodpecker/Yellow-crested Woodpecker_img2.j

📥 Downloading images for species 516 to 520...
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img1.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img2.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img3.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img4.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img5.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img6.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img7.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img8.jpg
Downloaded: ABC_BirdImages/Black-headed Gull/Black-headed Gull_img9.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img1.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img2.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img3.jpg
Downloaded: ABC_BirdImages/Diederik Cuckoo/Diederik Cuckoo_img4.jpg
Downloaded: ABC_BirdImages/Dieder

...

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 681 to 685...
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img1.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img2.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img3.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img4.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img5.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img6.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img7.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img8.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img9.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img10.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img11.jpg
Downloaded: ABC_BirdImages/White-necked Raven/White-necked Raven_img12.jpg
Downloaded: ABC_BirdImages/White-necke

📥 Downloading images for species 686 to 690...
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img1.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img2.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img3.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img4.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img5.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img6.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img7.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img8.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img9.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img10.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img11.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Raven_img12.jpg
Downloaded: ABC_BirdImages/Thick-billed Raven/Thick-billed Rav

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 691 to 695...
Downloaded: ABC_BirdImages/House Crow/House Crow_img1.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img2.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img3.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img4.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img5.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img6.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img7.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img8.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img9.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img10.jpg


In [13]:
# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-690, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 691st species (index 690)
    for start_idx in range(690, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking
        
        download_abc_images(abc_birds_df, start_idx, end_idx)  # Download batch
        
        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 690 + batch_size) % 10 == 0:  
            tqdm.write("⏳ Taking a 1-minute break...")
            time.sleep(timeout_duration)
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")

📥 Downloading images for species 691 to 695...
Downloaded: ABC_BirdImages/House Crow/House Crow_img1.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img2.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img3.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img4.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img5.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img6.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img7.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img8.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img9.jpg
Downloaded: ABC_BirdImages/House Crow/House Crow_img10.jpg
Downloaded: ABC_BirdImages/Rufous-naped Lark/Rufous-naped Lark_img1.jpg
Downloaded: ABC_BirdImages/Rufous-naped Lark/Rufous-naped Lark_img2.jpg
Downloaded: ABC_BirdImages/Rufous-naped Lark/Rufous-naped Lark_img3.jpg
Downloaded: ABC_BirdImages/Rufous-naped Lark/Rufous-naped Lark_img4.jpg
Downloaded: ABC_BirdImages/Rufous-naped Lark/Rufous-naped Lark_img5.jpg
Downloaded: 

📥 Downloading images for species 696 to 700...
Downloaded: ABC_BirdImages/Red-winged Lark/Red-winged Lark_img1.jpg
Downloaded: ABC_BirdImages/Red-winged Lark/Red-winged Lark_img2.jpg
Downloaded: ABC_BirdImages/Red-winged Lark/Red-winged Lark_img3.jpg
Downloaded: ABC_BirdImages/Red-winged Lark/Red-winged Lark_img4.jpg
Downloaded: ABC_BirdImages/Red-winged Lark/Red-winged Lark_img5.jpg
Downloaded: ABC_BirdImages/Red-winged Lark/Red-winged Lark_img6.jpg
Downloaded: ABC_BirdImages/Kidepo Lark/Kidepo Lark_img1.jpg
Downloaded: ABC_BirdImages/Plateau Lark/Plateau Lark_img1.jpg
Downloaded: ABC_BirdImages/Plateau Lark/Plateau Lark_img2.jpg
Downloaded: ABC_BirdImages/Plateau Lark/Plateau Lark_img3.jpg
Downloaded: ABC_BirdImages/Plateau Lark/Plateau Lark_img4.jpg
Downloaded: ABC_BirdImages/Somali Lark/Somali Lark_img1.jpg
Downloaded: ABC_BirdImages/Somali Lark/Somali Lark_img2.jpg
Downloaded: ABC_BirdImages/Somali Lark/Somali Lark_img3.jpg
Downloaded: ABC_BirdImages/Great Blue Turaco/Great Blue T

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 701 to 705...
Unexpected content type for Malachite Kingfisher: text/html; charset=UTF-8
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img2.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img3.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img4.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img5.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img6.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img7.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img8.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img9.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img10.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img11.jpg
Downloaded: ABC_BirdImages/Malachite Kingfisher/Malachite Kingfisher_img

📥 Downloading images for species 706 to 710...
Downloaded: ABC_BirdImages/Blue-shouldered Robin-Chat/Blue-shouldered Robin-Chat_img1.jpg
Downloaded: ABC_BirdImages/Blue-shouldered Robin-Chat/Blue-shouldered Robin-Chat_img2.jpg
Downloaded: ABC_BirdImages/Blue-shouldered Robin-Chat/Blue-shouldered Robin-Chat_img3.jpg
Downloaded: ABC_BirdImages/Blue-shouldered Robin-Chat/Blue-shouldered Robin-Chat_img4.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img1.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img2.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img3.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img4.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img5.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img6.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat/Chorister Robin-Chat_img7.jpg
Downloaded: ABC_BirdImages/Chorister Robin-Chat

...

📥 Downloading images for species 806 to 810...
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img1.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img2.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img3.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img4.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img5.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img6.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img7.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img8.jpg
Downloaded: ABC_BirdImages/Burchell's Courser/Burchell's Courser_img9.jpg
Downloaded: ABC_BirdImages/Somali Courser/Somali Courser_img1.jpg
Downloaded: ABC_BirdImages/Somali Courser/Somali Courser_img2.jpg
Downloaded: ABC_BirdImages/Somali Courser/Somali Courser_img3.jpg
Downloaded: ABC_BirdImages/Somali Courser/Somali Courser_img4.jpg
Downloaded: ABC_BirdIma

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 811 to 815...
Downloaded: ABC_BirdImages/Blue Cuckooshrike/Blue Cuckooshrike_img1.jpg
Downloaded: ABC_BirdImages/Blue Cuckooshrike/Blue Cuckooshrike_img2.jpg
Downloaded: ABC_BirdImages/Blue Cuckooshrike/Blue Cuckooshrike_img3.jpg
Downloaded: ABC_BirdImages/Comoro Blue Vanga/Comoro Blue Vanga_img1.jpg
Downloaded: ABC_BirdImages/Comoro Blue Vanga/Comoro Blue Vanga_img2.jpg
Downloaded: ABC_BirdImages/Comoro Blue Vanga/Comoro Blue Vanga_img3.jpg
Downloaded: ABC_BirdImages/Comoro Blue Vanga/Comoro Blue Vanga_img4.jpg
Downloaded: ABC_BirdImages/Madagascar Blue Vanga/Madagascar Blue Vanga_img1.jpg
Downloaded: ABC_BirdImages/Madagascar Blue Vanga/Madagascar Blue Vanga_img2.jpg
Downloaded: ABC_BirdImages/Madagascar Blue Vanga/Madagascar Blue Vanga_img3.jpg
Downloaded: ABC_BirdImages/Madagascar Blue Vanga/Madagascar Blue Vanga_img4.jpg
Downloaded: ABC_BirdImages/Blue-headed Sunbird/Blue-headed Sunbird_img1.jpg
Downloaded: ABC_BirdImages/Bl

📥 Downloading images for species 816 to 820...
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img1.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img2.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img3.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img4.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img5.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img6.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img7.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img8.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img9.jpg
Downloaded: ABC_BirdImages/Blue-throated Brown Sunbird/Blue-throated Brown Sunbird_img10.jpg
Downloaded: ABC_BirdImages/Olive

ConnectionError: ('Connection aborted.', TimeoutError(10060, 'A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond', None, 10060, None))

In [ ]:
# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-890, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 891st species (index 890)
    for start_idx in range(890, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking
        
        download_abc_images(abc_birds_df, start_idx, end_idx)  # Download batch
        
        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 890 + batch_size) % 10 == 0:  
            tqdm.write("⏳ Taking a 1-minute break...")
            time.sleep(timeout_duration)
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")

📥 Downloading images for species 891 to 895...
Downloaded: ABC_BirdImages/Cinereous Bunting/Cinereous Bunting_img1.jpg
Downloaded: ABC_BirdImages/Cinereous Bunting/Cinereous Bunting_img2.jpg
Downloaded: ABC_BirdImages/Cinereous Bunting/Cinereous Bunting_img3.jpg
Downloaded: ABC_BirdImages/Cirl Bunting/Cirl Bunting_img1.jpg
Downloaded: ABC_BirdImages/Cirl Bunting/Cirl Bunting_img2.jpg
Downloaded: ABC_BirdImages/Cirl Bunting/Cirl Bunting_img3.jpg
Downloaded: ABC_BirdImages/Cirl Bunting/Cirl Bunting_img4.jpg
Downloaded: ABC_BirdImages/Cirl Bunting/Cirl Bunting_img5.jpg
Downloaded: ABC_BirdImages/Golden-breasted Bunting/Golden-breasted Bunting_img1.jpg
Downloaded: ABC_BirdImages/Golden-breasted Bunting/Golden-breasted Bunting_img2.jpg
Downloaded: ABC_BirdImages/Golden-breasted Bunting/Golden-breasted Bunting_img3.jpg
Downloaded: ABC_BirdImages/Golden-breasted Bunting/Golden-breasted Bunting_img4.jpg
Downloaded: ABC_BirdImages/Golden-breasted Bunting/Golden-breasted Bunting_img5.jpg
Downloa

📥 Downloading images for species 896 to 900...
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img1.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img2.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img3.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img4.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img5.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img6.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img7.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img8.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img9.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img10.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img11.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img12.jpg
Downloaded: ABC_BirdImages/Lark-like Bunting/Lark-like Bunting_img13.jpg
Downloaded: A

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 901 to 905...
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img1.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img2.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img3.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img4.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img5.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img6.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img7.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img8.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img9.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bunting/Cinnamon-breasted Bunting_img10.jpg
Downloaded: ABC_BirdImages/Cinnamon-breasted Bun

📥 Downloading images for species 906 to 910...
Downloaded: ABC_BirdImages/Black-necked Eremomela/Black-necked Eremomela_img1.jpg
Downloaded: ABC_BirdImages/Black-necked Eremomela/Black-necked Eremomela_img2.jpg
Downloaded: ABC_BirdImages/Black-necked Eremomela/Black-necked Eremomela_img3.jpg
Downloaded: ABC_BirdImages/Black-necked Eremomela/Black-necked Eremomela_img4.jpg
Downloaded: ABC_BirdImages/Rufous-crowned Eremomela/Rufous-crowned Eremomela_img1.jpg
Downloaded: ABC_BirdImages/Rufous-crowned Eremomela/Rufous-crowned Eremomela_img2.jpg
Downloaded: ABC_BirdImages/Green-backed Eremomela/Green-backed Eremomela_img1.jpg
Downloaded: ABC_BirdImages/Yellow-vented Eremomela/Yellow-vented Eremomela_img1.jpg
Downloaded: ABC_BirdImages/Yellow-vented Eremomela/Yellow-vented Eremomela_img2.jpg
Downloaded: ABC_BirdImages/Yellow-vented Eremomela/Yellow-vented Eremomela_img3.jpg
Downloaded: ABC_BirdImages/Yellow-vented Eremomela/Yellow-vented Eremomela_img4.jpg
Downloaded: ABC_BirdImages/Karoo Er

...

✅ Resuming downloads...
📥 Downloading images for species 971 to 975...
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img1.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img2.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img3.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img4.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img5.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img6.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img7.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img8.jpg
Downloaded: ABC_BirdImages/Blue-throated Roller/Blue-throated Roller_img9.jpg
Downloaded: ABC_BirdImages/Dusky Twinspot/Dusky Twinspot_img1.jpg
Downloaded: ABC_BirdImages/Dusky Twinspot/Dusky Twinspot_img2.jpg
Downloaded: ABC_BirdImages/Dusky Twinspot/Dusky Twinspot_img3.jpg
Downloaded: ABC_BirdImages/Du

📥 Downloading images for species 976 to 980...
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img1.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img2.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img3.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img4.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img5.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img6.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img7.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img8.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img9.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img10.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img11.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img12.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img13.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img14.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Falcon_img15.jpg
Downloaded: ABC_BirdImages/Amur Falcon/Amur Fal

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 981 to 985...
Downloaded: ABC_BirdImages/Merlin/Merlin_img1.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img1.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img2.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img3.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img4.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img5.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img6.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img7.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img8.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img9.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img10.jpg
Downloaded: ABC_BirdImages/Sooty Falcon/Sooty Falcon_img11.jpg
Downloaded: ABC_BirdImages/African Hobby/African Hobby_img1.jpg
Downloaded: ABC_BirdImages/African Hobby/African Hobby_img2.jpg
Downloaded: ABC_BirdImages/African Hobby/African Hobby_img3.jpg
Dow

📥 Downloading images for species 986 to 990...
Downloaded: ABC_BirdImages/Taita Falcon/Taita Falcon_img1.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img1.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img2.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img3.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img4.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img5.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img6.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img7.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img8.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img9.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img10.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img11.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img12.jpg
Downloaded: ABC_BirdImages/Lesser Kestrel/Lesser Kestrel_img13.jpg
Downloaded: ABC_BirdImages/Le

In [ ]:
import time
from tqdm import tqdm

# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species - 1030, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 1031st species (index 1030)
    for start_idx in range(1030, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset

        tqdm.write(f"📥 Downloading images for species {start_idx + 1} to {end_idx}...")  # Progress tracking
        
        try:
            download_abc_images(abc_birds_df, start_idx, end_idx)  # Download batch
        except Exception as e:
            tqdm.write(f"❌ Error during downloading batch {start_idx + 1} to {end_idx}: {e}")
            continue  # Skip this batch and continue with the next one

        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar

        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 1030 + batch_size) % 10 == 0:  
            tqdm.write("⏳ Taking a 1-minute break...")
            time.sleep(timeout_duration)
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")


📥 Downloading images for species 1031 to 1035...
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img1.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img2.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img3.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img4.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img5.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img6.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img7.jpg
Downloaded: ABC_BirdImages/Eurasian Coot/Eurasian Coot_img8.jpg
Unexpected content type for Red-knobbed Coot: text/html; charset=UTF-8
Downloaded: ABC_BirdImages/Red-knobbed Coot/Red-knobbed Coot_img2.jpg
Downloaded: ABC_BirdImages/Red-knobbed Coot/Red-knobbed Coot_img3.jpg
Downloaded: ABC_BirdImages/Red-knobbed Coot/Red-knobbed Coot_img4.jpg
Downloaded: ABC_BirdImages/Red-knobbed Coot/Red-knobbed Coot_img5.jpg
Downloaded: ABC_BirdImages/Red-knobbed Coot/Red-knobbed Coot_img6.jpg
Downloaded: ABC_Bi

📥 Downloading images for species 1036 to 1040...
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img1.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img2.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img3.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img4.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img5.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img6.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img7.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img8.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img9.jpg
Downloaded: ABC_BirdImages/Large-billed Lark/Large-billed Lark_img10.jpg
Downloaded: ABC_BirdImages/Sun Lark/Sun Lark_img1.jpg
Downloaded: ABC_BirdImages/Sun Lark/Sun Lark_img2.jpg
Downloaded: ABC_BirdImages/Sun Lark/Sun Lark_img3.jpg
Downloaded: ABC_BirdImages/Sun Lark/Sun Lark_img4.jpg
Downloaded: AB

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1041 to 1045...
Downloaded: ABC_BirdImages/Madagascar Snipe/Madagascar Snipe_img1.jpg
Downloaded: ABC_BirdImages/Madagascar Snipe/Madagascar Snipe_img2.jpg
Downloaded: ABC_BirdImages/Madagascar Snipe/Madagascar Snipe_img3.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img1.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img2.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img3.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img4.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img5.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img6.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img7.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img8.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img9.jpg
Downloaded: ABC_BirdImages/Great Snipe/Great Snipe_img10.jpg
Downloaded: ABC_BirdImages/African Snipe/African Snipe_img1.jpg
Downloaded: ABC_BirdImages/African Snipe/African Sni

📥 Downloading images for species 1046 to 1050...
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img1.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img2.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img3.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img4.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img5.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img6.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img7.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img8.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img9.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img10.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turaco_img11.jpg
Downloaded: ABC_BirdImages/Purple-crested Turaco/Purple-crested Turac

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1051 to 1055...
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img2.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img3.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img4.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img5.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img6.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img7.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img8.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img9.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img10.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img11.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img12.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpeck

📥 Downloading images for species 1056 to 1060...
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img1.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img2.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img3.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img4.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img5.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img6.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img7.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img8.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img9.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img10.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img11.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img12.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img13.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img14.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img15.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img16.jpg
Downloaded: ABC_

❌ Error during downloading batch 1056 to 1060: ('Connection aborted.', TimeoutError(10060, 'A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond', None, 10060, None))
📥 Downloading images for species 1061 to 1065...
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img1.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img2.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img3.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img4.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img5.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img6.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img7.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img8.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img9.jpg
Downloaded: ABC_BirdImages/Rock Pratincole

In [ ]:
import time
import requests
from tqdm import tqdm

# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-1050, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 1051st species (index 1050)
    for start_idx in range(1050, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking

        try:
            download_abc_images(abc_birds_df, start_idx, end_idx)  # No timeout here!
        except Exception as e:
            tqdm.write(f"❌ Error: {e} - Skipping species {start_idx+1} to {end_idx}")

        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 1050 + batch_size) % 10 == 0:
            tqdm.write("⏳ Taking a 1-minute break...")
            
            # Sleep in small chunks so it's interruptible
            for _ in range(timeout_duration):  
                time.sleep(1)  # Allows interruptions every second
            
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")


📥 Downloading images for species 1051 to 1055...
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img1.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img2.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img3.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img4.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img5.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img6.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img7.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img8.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img9.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img10.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img11.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img12.jpg
Downloaded: ABC_BirdImages/Ground Woodpecker/Ground Woodpecker_img13.jpg
Downloaded:

📥 Downloading images for species 1056 to 1060...
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img1.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img2.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img3.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img4.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img5.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img6.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img7.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img8.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img9.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img10.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img11.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img12.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img13.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img14.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img15.jpg
Downloaded: ABC_BirdImages/Zebra Dove/Zebra Dove_img16.jpg
Downloaded: ABC_

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1061 to 1065...
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img1.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img2.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img3.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img4.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img5.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img6.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img7.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img8.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img9.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img10.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img11.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img12.jpg
Downloaded: ABC_BirdImages/Rock Pratincole/Rock Pratincole_img13.jpg
Downloaded: ABC_BirdImages/Rock Pratinc

📥 Downloading images for species 1066 to 1070...
Downloaded: ABC_BirdImages/Cinderella Waxbill/Cinderella Waxbill_img1.jpg
Downloaded: ABC_BirdImages/Cinderella Waxbill/Cinderella Waxbill_img2.jpg
Downloaded: ABC_BirdImages/Cinderella Waxbill/Cinderella Waxbill_img3.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img1.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img2.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img3.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img4.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img5.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img6.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img7.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img8.jpg
Downloaded: ABC_BirdImages/African Barred Owlet/African Barred Owlet_img9.jpg
Downloaded: ABC_BirdImages/

...

✅ Resuming downloads...
📥 Downloading images for species 1271 to 1275...
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img1.jpg
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img2.jpg
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img3.jpg
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img4.jpg
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img5.jpg
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img6.jpg
Downloaded: ABC_BirdImages/Mackinnon’s Shrike/Mackinnon’s Shrike_img7.jpg
Downloaded: ABC_BirdImages/Magpie Shrike/Magpie Shrike_img1.jpg
Downloaded: ABC_BirdImages/Magpie Shrike/Magpie Shrike_img2.jpg
Downloaded: ABC_BirdImages/Magpie Shrike/Magpie Shrike_img3.jpg
Downloaded: ABC_BirdImages/Magpie Shrike/Magpie Shrike_img4.jpg
Downloaded: ABC_BirdImages/Magpie Shrike/Magpie Shrike_img5.jpg
Downloaded: ABC_BirdImages/Magpie Shrike/Magpie Shrike_img6.jpg
Downloaded: ABC_BirdImage

📥 Downloading images for species 1276 to 1280...
Downloaded: ABC_BirdImages/Steppe Grey Shrike/Steppe Grey Shrike_img1.jpg
Downloaded: ABC_BirdImages/Steppe Grey Shrike/Steppe Grey Shrike_img2.jpg
Downloaded: ABC_BirdImages/Red-tailed Shrike/Red-tailed Shrike_img1.jpg
Downloaded: ABC_BirdImages/Red-tailed Shrike/Red-tailed Shrike_img2.jpg
Downloaded: ABC_BirdImages/Red-tailed Shrike/Red-tailed Shrike_img3.jpg
Downloaded: ABC_BirdImages/Red-tailed Shrike/Red-tailed Shrike_img4.jpg
Downloaded: ABC_BirdImages/Red-tailed Shrike/Red-tailed Shrike_img5.jpg
Unexpected content type for Woodchat Shrike: text/html; charset=UTF-8
Downloaded: ABC_BirdImages/Woodchat Shrike/Woodchat Shrike_img2.jpg
Downloaded: ABC_BirdImages/Woodchat Shrike/Woodchat Shrike_img3.jpg
Downloaded: ABC_BirdImages/Woodchat Shrike/Woodchat Shrike_img4.jpg
Downloaded: ABC_BirdImages/Woodchat Shrike/Woodchat Shrike_img5.jpg
Downloaded: ABC_BirdImages/Woodchat Shrike/Woodchat Shrike_img6.jpg
Downloaded: ABC_BirdImages/Woodch

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1281 to 1285...
Downloaded: ABC_BirdImages/Mew Gull/Mew Gull_img1.jpg
Downloaded: ABC_BirdImages/Mew Gull/Mew Gull_img2.jpg
Downloaded: ABC_BirdImages/Mew Gull/Mew Gull_img3.jpg
Downloaded: ABC_BirdImages/Ring-billed Gull/Ring-billed Gull_img1.jpg
Unexpected content type for Kelp Gull: text/html; charset=UTF-8
Unexpected content type for Kelp Gull: text/html; charset=UTF-8
Unexpected content type for Kelp Gull: text/html; charset=UTF-8
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img4.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img5.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img6.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img7.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img8.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img9.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img10.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img11.jpg
Downloaded: ABC_BirdImages/Kelp Gull/Kelp Gull_img12.

📥 Downloading images for species 1286 to 1290...
Downloaded: ABC_BirdImages/Great Black-backed Gull/Great Black-backed Gull_img1.jpg
Downloaded: ABC_BirdImages/Great Black-backed Gull/Great Black-backed Gull_img2.jpg
Downloaded: ABC_BirdImages/Great Black-backed Gull/Great Black-backed Gull_img3.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img1.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img2.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img3.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img4.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img5.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img6.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img7.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img8.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img9.jpg
Downloaded: ABC_BirdImages/Yellow

In [13]:
# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-1285, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 1286th species (index 1285)
    for start_idx in range(1285, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking

        try:
            download_abc_images(abc_birds_df, start_idx, end_idx)  # No timeout here!
        except Exception as e:
            tqdm.write(f"❌ Error: {e} - Skipping species {start_idx+1} to {end_idx}")

        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 1285 + batch_size) % 10 == 0:
            tqdm.write("⏳ Taking a 1-minute break...")
            
            # Sleep in small chunks so it's interruptible
            for _ in range(timeout_duration):  
                time.sleep(1)  # Allows interruptions every second
            
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")

📥 Downloading images for species 1286 to 1290...
Downloaded: ABC_BirdImages/Great Black-backed Gull/Great Black-backed Gull_img1.jpg
Downloaded: ABC_BirdImages/Great Black-backed Gull/Great Black-backed Gull_img2.jpg
Downloaded: ABC_BirdImages/Great Black-backed Gull/Great Black-backed Gull_img3.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img1.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img2.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img3.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img4.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img5.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img6.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img7.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img8.jpg
Downloaded: ABC_BirdImages/Yellow-legged Gull/Yellow-legged Gull_img9.jpg
Downloaded: ABC_BirdImages/Yellow

📥 Downloading images for species 1291 to 1295...
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img1.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img2.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img3.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img4.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img5.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img6.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img7.jpg
Downloaded: ABC_BirdImages/Cuckoo-roller/Cuckoo-roller_img8.jpg
Downloaded: ABC_BirdImages/Franklin's Gull/Franklin's Gull_img1.jpg
Downloaded: ABC_BirdImages/Franklin's Gull/Franklin's Gull_img2.jpg
Downloaded: ABC_BirdImages/Franklin's Gull/Franklin's Gull_img3.jpg
Downloaded: ABC_BirdImages/Franklin's Gull/Franklin's Gull_img4.jpg
Downloaded: ABC_BirdImages/Franklin's Gull/Franklin's Gull_img5.jpg
Downloaded: ABC_BirdImages/Franklin's Gull/Franklin's Gull_img6.jpg
Downloaded: ABC_BirdImages/Shor

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1296 to 1300...
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img1.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img2.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img3.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img4.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img5.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img6.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img7.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img8.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img9.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img10.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img11.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godwit_img12.jpg
Downloaded: ABC_BirdImages/Bar-tailed Godwit/Bar-tailed Godw

📥 Downloading images for species 1301 to 1305...
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img1.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img2.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img3.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img4.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img5.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img6.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img7.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img8.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img9.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img10.jpg
Downloaded: ABC_BirdImages/Hartlaub's Bustard/Hartlaub's Bustard_img11.jpg
Downloaded: ABC_BirdImages/Black-bellied Bustard/Black-bellied Bustard_img1.jpg
Downloaded: ABC_BirdImages/Black-bellied Bustard/Black-

...

❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/1778 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000002288CB5E310>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 1726 to 1730
📥 Downloading images for species 1731 to 1735...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/183 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000002288CB5EDF0>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 1731 to 1735
⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1736 to 1740...
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img1.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img2.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img3.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img4.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img5.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img6.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img7.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img8.jpg
Downloaded: ABC_BirdImages/Chestnut-crowned Sparrow-Weaver/Chestnut-crowned Sparrow-Weaver_img9.jpg
Downloaded: ABC_BirdImages/

📥 Downloading images for species 1741 to 1745...
Downloaded: ABC_BirdImages/Golden-naped Weaver/Golden-naped Weaver_img1.jpg
Downloaded: ABC_BirdImages/Golden-naped Weaver/Golden-naped Weaver_img2.jpg
Downloaded: ABC_BirdImages/Golden-naped Weaver/Golden-naped Weaver_img3.jpg
Downloaded: ABC_BirdImages/Cinnamon Weaver/Cinnamon Weaver_img1.jpg
Downloaded: ABC_BirdImages/Cinnamon Weaver/Cinnamon Weaver_img2.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img1.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img2.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img3.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img4.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img5.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img6.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img7.jpg
Downloaded: ABC_BirdImages/Baglafecht Weaver/Baglafecht Weaver_img8.jpg
Downloaded:

❌ Error: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)) - Skipping species 1741 to 1745
⏳ Taking a 1-minute break...


KeyboardInterrupt: 

In [13]:
# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-1415, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 1416th species (index 1415)
    for start_idx in range(1415, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking

        try:
            download_abc_images(abc_birds_df, start_idx, end_idx)  # No timeout here!
        except Exception as e:
            tqdm.write(f"❌ Error: {e} - Skipping species {start_idx+1} to {end_idx}")

        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 1415 + batch_size) % 10 == 0:
            tqdm.write("⏳ Taking a 1-minute break...")
            
            # Sleep in small chunks so it's interruptible
            for _ in range(timeout_duration):  
                time.sleep(1)  # Allows interruptions every second
            
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")

📥 Downloading images for species 1416 to 1420...
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img1.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img2.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img3.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img4.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img5.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img6.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img7.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img8.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img9.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-eater/Southern Carmine Bee-eater_img10.jpg
Downloaded: ABC_BirdImages/Southern Carmine Bee-ea

📥 Downloading images for species 1421 to 1425...
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img1.jpg
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img2.jpg
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img3.jpg
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img4.jpg
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img5.jpg
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img6.jpg
Downloaded: ABC_BirdImages/Somali Bee-eater/Somali Bee-eater_img7.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-eater_img1.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-eater_img2.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-eater_img3.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-eater_img4.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-eater_img5.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-eater_img6.jpg
Downloaded: ABC_BirdImages/Olive Bee-eater/Olive Bee-

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1426 to 1430...
Downloaded: ABC_BirdImages/White-breasted Mesite/White-breasted Mesite_img1.jpg
Downloaded: ABC_BirdImages/White-breasted Mesite/White-breasted Mesite_img2.jpg
Downloaded: ABC_BirdImages/White-breasted Mesite/White-breasted Mesite_img3.jpg
Downloaded: ABC_BirdImages/White-breasted Mesite/White-breasted Mesite_img4.jpg
Downloaded: ABC_BirdImages/White-breasted Mesite/White-breasted Mesite_img5.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img1.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img2.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img3.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img4.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img5.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img6.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img7.jpg
Downloaded: ABC_BirdImages/Reed Cormorant/Reed Cormorant_img8.jpg

❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Read timed out. (read timeout=10) - Skipping species 1426 to 1430
📥 Downloading images for species 1431 to 1435...
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img1.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img2.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img3.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img4.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img5.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img6.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img7.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img8.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img9.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img10.jpg
Downloaded: ABC_BirdImages/Yellow-billed Kite/Yellow-billed Kite_img11.j

...

✅ Resuming downloads...
📥 Downloading images for species 2146 to 2150...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/737 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB84884F0>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2146 to 2150
📥 Downloading images for species 2151 to 2155...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/2018 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB84884F0>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2151 to 2155
⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 2156 to 2160...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/2011 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB8488040>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2156 to 2160
📥 Downloading images for species 2161 to 2165...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/1766 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB84885B0>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2161 to 2165
⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 2166 to 2170...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/1767 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB8488520>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2166 to 2170
📥 Downloading images for species 2171 to 2175...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/8 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB8488280>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2171 to 2175
⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 2176 to 2180...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/2531 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB8488550>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2176 to 2180
📥 Downloading images for species 2181 to 2185...
❌ Error: HTTPSConnectionPool(host='www.africanbirdclub.org', port=443): Max retries exceeded with url: /afbid/search/browse/species/1601 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x0000017BB84884F0>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed')) - Skipping species 2181 to 2185
⏳ Taking a 1-minute break...


KeyboardInterrupt: 

In [14]:
# Define batch size
batch_size = 5
timeout_duration = 60  # 1 minute in seconds

# Get total number of species
total_species = len(abc_birds_df)

# Create a progress bar using tqdm
with tqdm(total=total_species-1820, desc="Downloading Images", unit="species", leave=True) as pbar:
    # Loop through the dataset in batches, starting from the 1821st species (index 1820)
    for start_idx in range(1820, total_species, batch_size):
        end_idx = min(start_idx + batch_size, total_species)  # Ensure we don't exceed the dataset
        
        tqdm.write(f"📥 Downloading images for species {start_idx+1} to {end_idx}...")  # Progress tracking

        try:
            download_abc_images(abc_birds_df, start_idx, end_idx)  # No timeout here!
        except Exception as e:
            tqdm.write(f"❌ Error: {e} - Skipping species {start_idx+1} to {end_idx}")

        pbar.update(min(batch_size, total_species - start_idx))  # Update progress bar
        
        # Introduce a 1-minute break after every two batches (10 species)
        if (start_idx - 1820 + batch_size) % 10 == 0:
            tqdm.write("⏳ Taking a 1-minute break...")
            
            # Sleep in small chunks so it's interruptible
            for _ in range(timeout_duration):  
                time.sleep(1)  # Allows interruptions every second
            
            tqdm.write("✅ Resuming downloads...")

tqdm.write("✅ All images downloaded successfully!")

📥 Downloading images for species 1821 to 1825...
Downloaded: ABC_BirdImages/Black-breasted Barbet/Black-breasted Barbet_img1.jpg
Downloaded: ABC_BirdImages/Black-breasted Barbet/Black-breasted Barbet_img2.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img1.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img2.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img3.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img4.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img5.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img6.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img7.jpg
Downloaded: ABC_BirdImages/Brown-headed Parrot/Brown-headed Parrot_img8.jpg
Downloaded: ABC_BirdImages/Yellow-fronted Parrot/Yellow-fronted Parrot_img1.jpg
Downloaded: ABC_BirdImages/Yellow-fronted Parrot/Yellow-fronted Parrot_img2.jpg
Downloaded: ABC_BirdIma

📥 Downloading images for species 1826 to 1830...
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img1.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img2.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img3.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img4.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img5.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img6.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img7.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img8.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img9.jpg
Downloaded: ABC_BirdImages/Meyer's Parrot/Meyer's Parrot_img10.jpg
Downloaded: ABC_BirdImages/Cape Parrot/Cape Parrot_img1.jpg
Downloaded: ABC_BirdImages/Cape Parrot/Cape Parrot_img2.jpg
Downloaded: ABC_BirdImages/Cape Parrot/Cape Parrot_img3.jpg
Downloaded: ABC_BirdImages/Cape Parrot/Cape Parrot_img4.jpg
Downloaded: ABC_BirdImages/Rüppell’s Parrot/Rüppel

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 1831 to 1835...
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img1.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img2.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img3.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img4.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img5.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img6.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img7.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img8.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img9.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img10.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img11.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img12.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img13.jpg
Downloaded: ABC_BirdImages/Martial Eagle/Martial Eagle_img14.jpg
Downloaded: ABC_BirdImages

📥 Downloading images for species 1836 to 1840...
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img1.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img2.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img3.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img4.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img5.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img6.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img7.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img8.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img9.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img10.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img11.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img12.jpg
Downloaded: ABC_BirdImages/Allen's Gallinule/Allen's Gallinule_img13.jpg
Downloaded:

...

⏳ Taking a 1-minute break...


✅ Resuming downloads...
📥 Downloading images for species 2361 to 2365...
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img1.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img2.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img3.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img4.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img5.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img6.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img7.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img8.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img9.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img10.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img11.jpg
Downloaded: ABC_BirdImages/Ethiopian White-eye/Ethiopian White-eye_img12.jpg
Downloaded: 

📥 Downloading images for species 2366 to 2369...
Downloaded: ABC_BirdImages/Green White-eye/Green White-eye_img1.jpg
Downloaded: ABC_BirdImages/Green White-eye/Green White-eye_img2.jpg
Downloaded: ABC_BirdImages/Pemba White-eye/Pemba White-eye_img1.jpg
Downloaded: ABC_BirdImages/Pemba White-eye/Pemba White-eye_img2.jpg
Downloaded: ABC_BirdImages/Pemba White-eye/Pemba White-eye_img3.jpg
Downloaded: ABC_BirdImages/Pemba White-eye/Pemba White-eye_img4.jpg
Downloaded: ABC_BirdImages/Pemba White-eye/Pemba White-eye_img5.jpg
Downloaded: ABC_BirdImages/Pemba White-eye/Pemba White-eye_img6.jpg
Unexpected content type for Cape White-eye: text/html; charset=UTF-8
Downloaded: ABC_BirdImages/Cape White-eye/Cape White-eye_img2.jpg
Downloaded: ABC_BirdImages/Cape White-eye/Cape White-eye_img3.jpg
Downloaded: ABC_BirdImages/Cape White-eye/Cape White-eye_img4.jpg
Downloaded: ABC_BirdImages/Cape White-eye/Cape White-eye_img5.jpg
Downloaded: ABC_BirdImages/Cape White-eye/Cape White-eye_img6.jpg
Download

⏳ Taking a 1-minute break...


✅ Resuming downloads...
✅ All images downloaded successfully!


In [13]:
df = abc_birds_df
download_abc_images(df, 1240, 1260)

Downloaded: ABC_BirdImages/Gabela Bushshrike/Gabela Bushshrike_img1.jpg
Downloaded: ABC_BirdImages/Gabela Bushshrike/Gabela Bushshrike_img2.jpg
Downloaded: ABC_BirdImages/Gabela Bushshrike/Gabela Bushshrike_img3.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img1.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img2.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img3.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img4.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img5.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img6.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img7.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img8.jpg
Downloaded: ABC_BirdImages/Crimson-breasted Shrike/Crimson-breasted Shrike_img9.jpg
Downloaded: ABC_BirdImages/C

In [14]:
# Define the image directory
image_dir = "ABC_BirdImages"

# Get list of species folders and sort alphabetically
species_folders = sorted([folder for folder in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, folder))])

# Assign each species a unique index
species_mapping = {species: idx + 1 for idx, species in enumerate(species_folders)}

# Display the mapping
print(species_mapping)

{'Abbott’s Starling': 1, "Abdim's Stork": 2, 'Aberdare Cisticola': 3, 'Abyssinian Catbird': 4, 'Abyssinian Crimsonwing': 5, 'Abyssinian Ground Hornbill': 6, 'Abyssinian Ground Thrush': 7, 'Abyssinian Longclaw': 8, 'Abyssinian Owl': 9, 'Abyssinian Roller': 10, 'Abyssinian Scimitarbill': 11, 'Abyssinian Slaty Flycatcher': 12, 'Abyssinian Thrush': 13, 'Abyssinian Wheatear': 14, 'Abyssinian White-eye': 15, 'Abyssinian Woodpecker': 16, 'Acacia Pied Barbet': 17, 'Acacia Tit': 18, 'Adamawa Turtle Dove': 19, 'Afep Pigeon': 20, 'African Barred Owlet': 21, 'African Black Duck': 22, 'African Black Swift': 23, 'African Blue Flycatcher': 24, 'African Blue Tit': 25, 'African Broadbill': 26, 'African Chaffinch': 27, 'African Citril': 28, 'African Collared Dove': 29, 'African Crake': 30, 'African Crimson-winged Finch': 31, 'African Cuckoo': 32, 'African Cuckoo-Hawk': 33, 'African Darter': 34, 'African Desert Warbler': 35, 'African Dusky Flycatcher': 36, 'African Dwarf Kingfisher': 37, 'African Emerald

In [15]:
species_images = {}

# Iterate over species folders
for species_folder in species_folders:
    species_path = os.path.join(image_dir, species_folder)

    # Get image files, sorted by name
    image_files = sorted([f for f in os.listdir(species_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])

    # Store in dictionary
    species_images[species_folder] = image_files

# Display first species and its images
first_species = list(species_images.keys())[0]
print(f"{first_species}: {species_images[first_species]}")

Abbott’s Starling: ['Abbott’s Starling_img1.jpg', 'Abbott’s Starling_img2.jpg', 'Abbott’s Starling_img3.jpg', 'Abbott’s Starling_img4.jpg', 'Abbott’s Starling_img5.jpg', 'Abbott’s Starling_img6.jpg', 'Abbott’s Starling_img7.jpg', 'Abbott’s Starling_img8.jpg']


In [16]:
new_rows = []

# Iterate over species
for species_folder in species_folders:
    species_index = species_mapping[species_folder]
    image_files = species_images[species_folder]

    # Track expected images
    expected_count = len(image_files) if image_files else 0

    for img_idx in range(1, expected_count + 1):
        # Generate expected filename (e.g., 1001.jpg)
        new_file_name = f"{species_index}{img_idx:03d}.jpg"

        # Find matching file or mark as missing
        matching_files = [f for f in image_files if f"_{img_idx}." in f]

        if matching_files:
            actual_file = matching_files[0]
            file_path = os.path.join(image_dir, species_folder, actual_file)
        else:
            # Missing image case
            actual_file = "UTF-8"
            file_path = None  

        # Append row
        new_rows.append([species_folder, new_file_name, file_path])

# Display first few rows
print(new_rows[:10])

[['Abbott’s Starling', '1001.jpg', None], ['Abbott’s Starling', '1002.jpg', None], ['Abbott’s Starling', '1003.jpg', None], ['Abbott’s Starling', '1004.jpg', None], ['Abbott’s Starling', '1005.jpg', None], ['Abbott’s Starling', '1006.jpg', None], ['Abbott’s Starling', '1007.jpg', None], ['Abbott’s Starling', '1008.jpg', None], ["Abdim's Stork", '2001.jpg', None], ["Abdim's Stork", '2002.jpg', None]]


In [19]:
# Convert new rows into a DataFrame
columns = ["Scientific_Name", "File_Name", "Image_Path"]
new_df = pd.DataFrame(new_rows, columns=columns)

# Merge with existing df
final_abd_birds_df = abc_birds_df.merge(new_df, on="Scientific_Name", how="left")

# Display result
print(final_abd_birds_df.head())

              Scientific_Name                  Common_Name  Image_Count  \
0           Accipiter henstii              Henst’s Goshawk            4   
1  Accipiter madagascariensis       Madagascar Sparrowhawk            5   
2             Accipiter nisus         Eurasian Sparrowhawk            6   
3        Accipiter ovampensis           Ovambo Sparrowhawk            8   
4       Accipiter rufiventris  Rufous-breasted Sparrowhawk            4   

                         Image_Link File_Name Image_Path  
0  /afbid/search/browse/species/290       NaN        NaN  
1  /afbid/search/browse/species/285       NaN        NaN  
2  /afbid/search/browse/species/286       NaN        NaN  
3  /afbid/search/browse/species/284       NaN        NaN  
4  /afbid/search/browse/species/287       NaN        NaN  
